# 🛡️ Bray PPE Monitor 2.0 — End-to-End Federated Learning Pipeline
### Multi-Zone Real-Time PPE Compliance with YOLOv8 & Flower (`flwr`)
**Tested & Verified Locally: Ready for Google Colab GPU (T4)**

This notebook executes the complete training and evaluation pipeline:
1. **GPU & Git LFS Setup**: Verifies GPU and downloads the baseline `backend/best.pt` (67MB).
2. **Dataset Download**: Downloads the verified Roboflow PPE dataset (75MB zip) via direct signed URL.
3. **Partitioning & Polygon-to-Box Conversion**: Partitions data into **Zone A** and **Zone B**, converts polygon segments to bounding boxes, clamps coordinates, and remaps classes to the 11-class single source of truth.
4. **Solo Baselines**: Trains Node A and Node B independently (15 epochs) to capture cross-zone degradation.
5. **Federated Training (FedAvg)**: Runs Flower server + 2 clients across 5 rounds.
6. **Cross-Zone 4-Way Evaluation**: Validates performance across both zones, producing `comparison_table.csv`.
7. **OpenVINO Re-Export**: Converts federated global model to OpenVINO IR for live inference.
8. **Artifact Packaging**: Bundles all deliverables into `federated_artifacts.zip` and downloads to your PC.


In [ ]:
# Step 1: Hardware Check (Expect NVIDIA T4 GPU)
!nvidia-smi


In [ ]:
# Step 2: Install dependencies
!pip install -q "ultralytics>=8.3.0,<9.0.0" "flwr>=1.5.0" "openvino>=2024.0.0" pandas


In [ ]:
# Step 3: Clone Repository, install git-lfs, and pull binary baseline weights
import os, hashlib, sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/Kailash-Murali/ppe-project-2.0.git"
REPO_DIR = Path("/content/ppe-project-2.0")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Install git-lfs on Linux and pull the actual 67MB weights
!apt-get update -qq && apt-get install -y -qq git-lfs
!git lfs install
!git lfs pull

base_pt = REPO_DIR / "backend" / "best.pt"

# Fallback: If Git LFS ever fails over network, prompt upload
if base_pt.exists() and base_pt.stat().st_size < 1000000:
    print("⚠️ Git LFS did not pull full weights. Please upload your local backend/best.pt:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.pt'):
            shutil.move(fname, str(base_pt))
            print(f"Replaced with uploaded {fname}")

assert base_pt.exists(), f"Base checkpoint not found at {base_pt}"

# Verify SHA256 (case-insensitive)
hasher = hashlib.sha256()
with open(base_pt, "rb") as f:
    while chunk := f.read(8192 * 1024):
        hasher.update(chunk)
current_hash = hasher.hexdigest().lower()
file_mb = round(base_pt.stat().st_size / (1024 * 1024), 2)
print(f"Current best.pt SHA256: {current_hash} ({file_mb} MB)")

hash_file = REPO_DIR / "CHECKPOINT_HASH.txt"
if hash_file.exists():
    expected_hash = hash_file.read_text().strip().split()[0].lower()
    print(f"Expected SHA256:       {expected_hash}")
    assert current_hash == expected_hash, f"Hash mismatch! Got {current_hash}, expected {expected_hash}"
    print("✅ Baseline checkpoint verified against CHECKPOINT_HASH.txt!")


### Step 4: Download Roboflow Dataset (Direct Signed Zip URL)
Downloads the exact 75 MB dataset archive directly from Roboflow's CDN and extracts it into `/content/dataset_raw`.


In [ ]:
# Step 4: Direct Download & Unpack Roboflow Dataset
import os, shutil
from pathlib import Path

DATA_RAW = Path("/content/dataset_raw")
shutil.rmtree(DATA_RAW, ignore_errors=True)
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Direct Roboflow export URL (verified 78.9 MB archive)
print("Downloading Roboflow dataset (~75 MB)...")
!curl -L -o /content/dataset.zip "https://app.roboflow.com/ds/e9cqZF04cL?key=HuL8u4PO3Y"

print("Unpacking dataset into /content/dataset_raw...")
!unzip -q /content/dataset.zip -d /content/dataset_raw
!rm /content/dataset.zip

train_count = len(list((DATA_RAW / "train" / "images").glob("*.*")))
val_count = len(list((DATA_RAW / "valid" / "images").glob("*.*")))
print(f"✅ Dataset ready: {train_count} train images, {val_count} val images.")


### Step 5: Non-IID Zone Partitioning, Polygon Conversion & Class Alignment
- Partitions images 50/50 into **Zone A** (`node_a`) and **Zone B** (`node_b`).
- Pre-indexes all label files by name to guarantee 100% stem matching (avoiding glob wildcard bugs).
- Converts polygon segmentations into standard YOLO bounding boxes.
- Clamps coordinates to `[0.001, 0.999]` so YOLO never complains about negative/out-of-bounds boxes.
- Remaps Roboflow class `4` (`person`) to model class `9` (`person`).
- Writes clean `data.yaml` configs matching our 11-class single source of truth.


In [ ]:
# Step 5: Verified Non-IID Partitioning & Label Conversion
import os, random, yaml, shutil
from pathlib import Path

random.seed(42)

raw_base = Path("/content/dataset_raw")
repo_dir = Path("/content/ppe-project-2.0")
target_base = repo_dir / "federated" / "data"

train_imgs = sorted(list((raw_base / "train" / "images").glob("*.*")))
val_imgs = sorted(list((raw_base / "valid" / "images").glob("*.*")))

assert len(train_imgs) > 0, "No train images found in dataset_raw! Check Step 4."
assert len(val_imgs) > 0, "No val images found in dataset_raw! Check Step 4."

split_t = len(train_imgs) // 2
split_v = len(val_imgs) // 2

zone_splits = {
    "node_a": {"train": train_imgs[:split_t], "val": val_imgs[:split_v]},
    "node_b": {"train": train_imgs[split_t:], "val": val_imgs[split_v:]}
}

CLASS_REMAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 9}

names = {
    0: "boots", 1: "glasses", 2: "gloves", 3: "helmet",
    4: "no boots", 5: "no glasses", 6: "no gloves", 7: "no helmet",
    8: "no vest", 9: "person", 10: "vest"
}

def convert_line(parts):
    try:
        orig_cls = int(parts[0])
        new_cls = CLASS_REMAP.get(orig_cls, orig_cls)
        
        if len(parts) == 5:
            xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        elif len(parts) > 5:
            coords = [float(x) for x in parts[1:]]
            xs = coords[0::2]
            ys = coords[1::2]
            if not xs or not ys: return None
            min_x, max_x = min(xs), max(xs)
            min_y, max_y = min(ys), max(ys)
            w = max_x - min_x
            h = max_y - min_y
            xc = min_x + w / 2.0
            yc = min_y + h / 2.0
        else:
            return None
            
        # Strict normalization clamp [0.001, 0.999]
        w = max(0.005, min(0.999, w))
        h = max(0.005, min(0.999, h))
        xc = max(w / 2.0, min(1.0 - w / 2.0, xc))
        yc = max(h / 2.0, min(1.0 - h / 2.0, yc))
        return f"{new_cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
    except:
        return None

# Pre-index all label files by filename
raw_labels = {p.name: p for p in raw_base.rglob("*.txt")}
print(f"Indexed {len(raw_labels)} raw label files.")

for node_name, splits in zone_splits.items():
    node_path = target_base / node_name
    for split, img_list in splits.items():
        dst_img_dir = node_path / "images" / split
        dst_lbl_dir = node_path / "labels" / split
        
        shutil.rmtree(dst_img_dir, ignore_errors=True)
        shutil.rmtree(dst_lbl_dir, ignore_errors=True)
        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        box_count = 0
        for img_p in img_list:
            shutil.copy2(img_p, dst_img_dir / img_p.name)
            txt_name = f"{img_p.stem}.txt"
            dst_lbl = dst_lbl_dir / txt_name
            src_lbl = raw_labels.get(txt_name)
            
            if src_lbl and src_lbl.exists():
                lines = src_lbl.read_text(encoding="utf-8").splitlines()
                boxes = [convert_line(l.strip().split()) for l in lines if l.strip()]
                boxes = [b for b in boxes if b is not None]
                dst_lbl.write_text("\n".join(boxes), encoding="utf-8")
                box_count += len(boxes)
            else:
                dst_lbl.write_text("", encoding="utf-8")
                
        print(f"[{node_name}] {split}: {len(img_list)} images, populated with {box_count} valid bounding boxes.")

    # Write data.yaml with absolute path
    yaml_content = {
        "path": str(node_path.resolve()),
        "train": "images/train",
        "val": "images/val",
        "nc": 11,
        "names": names
    }
    with open(node_path / "data.yaml", "w", encoding="utf-8") as f:
        yaml.dump(yaml_content, f, sort_keys=False)
    print(f"✅ Generated {node_path / 'data.yaml'}")

# Clear any old Ultralytics cache files
for c in repo_dir.rglob("*.cache"):
    c.unlink()

print("\n" + "="*60)
print("🎉 Zone partitioning and polygon-to-box conversion successful!")
print("Cleared old label caches. Ready to run Step 6!")
print("="*60)


### Step 6: Train Solo Baselines (Node A & Node B)
Train Zone A and Zone B independently (15 epochs each, batch=8, imgsz=640) fine-tuning from `backend/best.pt`.
- Node A output: `fed_baseline/nodeA_local/weights/best.pt`
- Node B output: `fed_baseline/nodeB_local/weights/best.pt`


In [ ]:
%cd {REPO_DIR}/federated

print("==================================================================")
print("TRAINING SOLO BASELINE: NODE A (15 Epochs on Zone A data)")
print("==================================================================")
!yolo detect train model=../backend/best.pt data=data/node_a/data.yaml epochs=15 imgsz=640 batch=8 seed=42 device=0 name=nodeA_local project=../fed_baseline exist_ok=True

print("\n==================================================================")
print("TRAINING SOLO BASELINE: NODE B (15 Epochs on Zone B data)")
print("==================================================================")
!yolo detect train model=../backend/best.pt data=data/node_b/data.yaml epochs=15 imgsz=640 batch=8 seed=42 device=0 name=nodeB_local project=../fed_baseline exist_ok=True

print("✅ Both solo baseline checkpoints trained successfully!")


### Step 7: Federated Training (Flower FedAvg)
We launch the Flower server (`fed_server.py`) and launch two client processes (`fed_client.py`) simultaneously.
Across 5 rounds, each client trains for 3 epochs locally and submits updated weights.
The server aggregates parameters with FedAvg and writes:
- `federated/federated_global.pt`
- `federated/fedavg_metrics.csv`


In [ ]:
import subprocess, time, sys

%cd {REPO_DIR}/federated

server_cmd = [
    sys.executable, "fed_server.py",
    "--rounds", "5",
    "--min-clients", "2",
    "--port", "8088",
    "--output", "federated_global.pt",
    "--log-csv", "fedavg_metrics.csv"
]

client_a_cmd = [
    sys.executable, "fed_client.py",
    "--data", "data/node_a/data.yaml",
    "--node-id", "node_a",
    "--server", "127.0.0.1:8088",
    "--device", "0"
]

client_b_cmd = [
    sys.executable, "fed_client.py",
    "--data", "data/node_b/data.yaml",
    "--node-id", "node_b",
    "--server", "127.0.0.1:8088",
    "--device", "0"
]

print("Starting Flower FedAvg Server...")
server_proc = subprocess.Popen(server_cmd)
time.sleep(4)  # Wait for server to bind

print("Starting Node A and Node B clients...")
client_a_proc = subprocess.Popen(client_a_cmd)
client_b_proc = subprocess.Popen(client_b_cmd)

# Wait for federated training to finish
client_a_proc.wait()
client_b_proc.wait()
server_proc.wait()

print("\n✅ Federated training complete!")
print("Saved federated_global.pt and fedavg_metrics.csv.")


### Step 8: 4-Way Cross-Zone Evaluation Matrix
Evaluate each model on both Zone A and Zone B validation sets:
1. Original Base Model (`backend/best.pt`)
2. Node A Solo Model (`nodeA_local`)
3. Node B Solo Model (`nodeB_local`)
4. FedAvg Global Model (`federated_global.pt`)

Saves output to `comparison_table.csv`.


In [ ]:
%cd {REPO_DIR}/federated

!python eval_comparison.py \
  --zone-a data/node_a/data.yaml \
  --zone-b data/node_b/data.yaml \
  --solo-a ../fed_baseline/nodeA_local/weights/best.pt \
  --solo-b ../fed_baseline/nodeB_local/weights/best.pt \
  --fedavg federated_global.pt \
  --base ../backend/best.pt \
  --output-csv comparison_table.csv

import pandas as pd
df = pd.read_csv("comparison_table.csv")
print("\n" + "="*75)
print("FINAL CROSS-ZONE EVALUATION MATRIX")
print("="*75)
display(df)


### Step 9: Re-Export Federated Model to OpenVINO IR
Converts `federated_global.pt` to OpenVINO IR (`best.xml`, `best.bin`, `metadata.yaml`) and deploys it into `backend/best_openvino_model/`.
This file format provides high-throughput GPU/iGPU inference on your local machine.


In [ ]:
%cd {REPO_DIR}/federated

!python export_to_openvino.py \
  --model federated_global.pt \
  --target ../backend/best_openvino_model

print("✅ OpenVINO model exported to backend/best_openvino_model!")


### Step 10: Bundle All Deliverables & Download ZIP
Zips all generated model weights, metrics CSVs, and OpenVINO IR files into `federated_artifacts.zip` and triggers automatic browser download.


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

zip_path = Path("/content/federated_artifacts.zip")

print("Packaging artifacts into zip...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    # 1. Federated Global PyTorch model
    fed_pt = REPO_DIR / "federated" / "federated_global.pt"
    if fed_pt.exists():
        zipf.write(fed_pt, arcname="federated/federated_global.pt")
        print(" + federated/federated_global.pt")
        
    # 2. Evaluation CSVs
    for csv_name in ["comparison_table.csv", "fedavg_metrics.csv"]:
        csv_file = REPO_DIR / "federated" / csv_name
        if csv_file.exists():
            zipf.write(csv_file, arcname=f"federated/{csv_name}")
            print(f" + federated/{csv_name}")
            
    # 3. Solo Baselines
    for node in ["nodeA_local", "nodeB_local"]:
        solo_pt = REPO_DIR / "fed_baseline" / node / "weights" / "best.pt"
        if solo_pt.exists():
            zipf.write(solo_pt, arcname=f"fed_baseline/{node}/weights/best.pt")
            print(f" + fed_baseline/{node}/weights/best.pt")
            
    # 4. OpenVINO IR Model for Live Inference
    vino_dir = REPO_DIR / "backend" / "best_openvino_model"
    if vino_dir.exists():
        for item in vino_dir.iterdir():
            if item.is_file():
                zipf.write(item, arcname=f"backend/best_openvino_model/{item.name}")
                print(f" + backend/best_openvino_model/{item.name}")

print(f"\n✅ Created archive: {zip_path} ({round(zip_path.stat().st_size / (1024*1024), 2)} MB)")
print("Downloading federated_artifacts.zip to your computer...")
files.download(str(zip_path))


### Step 11: Frontend Metrics Snippet Generator
Run this cell to generate the exact code to paste into `frontend/src/app/components/ModelMetrics/ModelMetrics.tsx`.


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = REPO_DIR / "federated" / "comparison_table.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    
    fed_rows = df[df["Model"] == "FedAvg Global Model"]
    base_rows = df[df["Model"] == "Original Base Model"]
    
    if not fed_rows.empty and not base_rows.empty:
        fed_map50 = round(fed_rows["mAP50"].mean() * 100, 1)
        fed_map50_95 = round(fed_rows["mAP50-95"].mean() * 100, 1)
        fed_prec = round(fed_rows["Precision"].mean() * 100, 1)
        fed_rec = round(fed_rows["Recall"].mean() * 100, 1)
        
        base_map50 = round(base_rows["mAP50"].mean() * 100, 1)
        base_map50_95 = round(base_rows["mAP50-95"].mean() * 100, 1)
        base_prec = round(base_rows["Precision"].mean() * 100, 1)
        base_rec = round(base_rows["Recall"].mean() * 100, 1)
        
        print("// ================================================================")
        print("// COPY & PASTE THIS INTO ModelMetrics.tsx (lines 47-52)")
        print("// ================================================================")
        print(f"const OVERALL_METRICS: OverallMetric[] = [")
        print(f"  {{ label: 'mAP50',      unit: '%', baseline: {base_map50}, federated: {fed_map50} }},")
        print(f"  {{ label: 'mAP50-95',  unit: '%', baseline: {base_map50_95}, federated: {fed_map50_95} }},")
        print(f"  {{ label: 'Precision', unit: '%', baseline: {base_prec}, federated: {fed_prec} }},")
        print(f"  {{ label: 'Recall',    unit: '%', baseline: {base_rec}, federated: {fed_rec} }},")
        print(f"];\n")
        
        print("// ================================================================")
        print("// UPDATE ZONE_ROWS IN ModelMetrics.tsx (lines 96-102)")
        print("// ================================================================")
        print("const ZONE_ROWS: ZoneRow[] = [")
        for m_name in ["Original Base Model", "Node A Solo Model", "Node B Solo Model", "FedAvg Global Model"]:
            m_df = df[df["Model"] == m_name]
            za = round(m_df[m_df["Evaluation Zone"].str.contains("A")]["mAP50"].values[0] * 100, 1) if not m_df[m_df["Evaluation Zone"].str.contains("A")].empty else None
            zb = round(m_df[m_df["Evaluation Zone"].str.contains("B")]["mAP50"].values[0] * 100, 1) if not m_df[m_df["Evaluation Zone"].str.contains("B")].empty else None
            avg = round((za + zb) / 2.0, 1) if za and zb else None
            label = "Baseline (best.pt)" if "Base" in m_name else ("Node A Solo" if "Solo Model A" in m_name or "Node A Solo" in m_name else ("Node B Solo" if "Node B Solo" in m_name else "FedAvg Global"))
            print(f"  {{ model: '{label}', zoneA: {za}, zoneB: {zb}, crossZoneAvg: {avg}, status: 'done' }},")
        print("];")
